In [0]:
spark.conf.set(
  "fs.azure.account.key.intechsg30.dfs.core.windows.net",
  "keyval"
)

dbutils.fs.ls("abfss://bronze@intechsg30.dfs.core.windows.net/")

In [0]:
dbutils.fs.ls("abfss://gold@intechsg30.dfs.core.windows.net/")

In [0]:
df = spark.read.format('delta').load("abfss://silver@intechsg30.dfs.core.windows.net/SalesLT/Address")
display(df)

In [0]:
from pyspark.sql.functions import col

def rename_columns_to_snake_case(df):
    """
    convert col names from PascalCase to snake_case
    args:
        df: Spark DataFrame
    return:
        df: Spark DataFrame
    """

    column_names = df.columns

    rename_map={}

    for old_col_name in column_names:
        new_col_name = "".join([
            "_" + char.lower() if (
                char.isupper()
                and idx >0
                and not old_col_name[idx-1].isupper()
            ) else char.lower()
            for idx, char in enumerate(old_col_name)
        ]).lstrip("_")

        if new_col_name in rename_map.values():
            raise ValueError(f"Duplicate column name found after renaming: {new_col_name}")

        rename_map[old_col_name] = new_col_name

    for old_col_name, new_col_name in rename_map.items():
        df = df.withColumnRenamed(old_col_name, new_col_name)
    return df

In [0]:
df = rename_columns_to_snake_case(df)
display(df)

In [0]:
table_name_temp = []

for i in dbutils.fs.ls("abfss://silver@intechsg30.dfs.core.windows.net/SalesLT/Address"):
    table_name_temp.append(i)
table_name_temp

In [0]:
table_name=[]
for i in dbutils.fs.ls("abfss://silver@intechsg30.dfs.core.windows.net/SalesLT"):
    table_name.append(i.name.split('/')[0])

In [0]:
table_name

In [0]:
for name in table_name:
    path = "abfss://silver@intechsg30.dfs.core.windows.net/SalesLT/" + name
    df = spark.read.format('delta').load(path)
    df = rename_columns_to_snake_case(df)

    output_path = "abfss://gold@intechsg30.dfs.core.windows.net/SalesLT/" + name+"/"
    df.write.mode("overwrite").format("delta").save(output_path)

In [0]:
display(df)

In [0]:
for table in ["Address", "Customer", "CustomerAddress", "Product", "ProductCategory", 
              "ProductDescription", "ProductModel", "ProductModelProductDescription",
              "SalesOrderDetail", "SalesOrderHeader"]:
    path = f"abfss://gold@intechsg30.dfs.core.windows.net/SalesLT/{table}"
    df = spark.read.format("delta").load(path)
    df.write.format("delta") \
        .option("delta.minReaderVersion", "1") \
        .option("delta.minWriterVersion", "2") \
        .mode("overwrite") \
        .save(path)
    print(f"Rewrote {table}")